In [ ]:
from pymongo import MongoClient
from openai import AzureOpenAI
import pandas as pd

client = MongoClient("mongodb://localhost:27017/")
db = client["yelp_business"]
collection = db["business"]

business_docs = list(collection.find())  # 你也可以用分页处理
client = AzureOpenAI(
    api_key="609ced4d971240b8a08f7fb0e6d846ea",
    api_version="2024-08-01-preview",
    azure_endpoint="https://promptdelta-nc.openai.azure.com",
)
deployment_name = "gpt-4o-mini"
def is_located_in_indianapolis(description):
    prompt = (
        f"Does the following business description clearly indicate that the business is located in Indianapolis?\n\n"
        f"Description: {description}\n\n"
        "Please answer only 'yes' or 'no'."
    )

    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You are a helpful assistant that determines the city location of a business from its description."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=5
        )
        return response.choices[0].message.content.strip().lower() == "yes"
    except Exception as e:
        print(f"❌ GPT error: {e}")
        return False
indianapolis_businesses = []

for i, biz in enumerate(business_docs):
    desc = biz.get("description", "")
    if not desc:
        continue

    result = is_located_in_indianapolis(desc)
    if result:
        indianapolis_businesses.append(biz)
        print(f"✅ Found: Business #{i}, {desc} → Indianapolis")
    else:
        print(f"❌ Skip: Business #{i} → Not in Indianapolis")
df_indianapolis = pd.DataFrame(indianapolis_businesses)
print(f"\n Retained {len(df_indianapolis)} businesses in Indianapolis")

In [ ]:
import duckdb

# 连接 DuckDB 数据库
con = duckdb.connect("../query_dataset/yelp_user.db")

# 列出所有表
tables = con.execute("SHOW TABLES").fetchall()

print("📋 数据库中包含的表：")
for t in tables:
    print("-", t[0])

df_review = con.execute("SELECT * FROM review").fetchdf()
df_tip = con.execute("SELECT * FROM tip").fetchdf()
df_user = con.execute("SELECT * FROM user").fetchdf()
con.close()

In [ ]:
import json
import duckdb
from pymongo import MongoClient
from openai import AzureOpenAI

deployment_name = "gpt-4o-mini"
client = AzureOpenAI(
    api_key="609ced4d971240b8a08f7fb0e6d846ea",
    api_version="2024-08-01-preview",
    azure_endpoint="https://promptdelta-nc.openai.azure.com"
)

# ========== Step 1: 加载两个 DuckDB 表中的 ID ==========
# business_ref from yelp_user.db
con_duck = duckdb.connect("../query_dataset/yelp_user.db")
business_refs = con_duck.execute("SELECT DISTINCT business_ref FROM review").fetchdf()["business_ref"].dropna().tolist()

# ========== Step 2: 从 MongoDB 读取 business_id ==========
client_mongo = MongoClient("mongodb://localhost:27017/")
business_ids = client_mongo["yelp_business"]["business"].distinct("business_id")

# ========== Step 2: 用 GPT 识别它们之间的映射规则 ==========
def get_mapping_rule(business_ids, business_refs):
    prompt = (
        "You are given two complete ID columns from two different datasets:\n"
        f"- The first column is `business_ref` from a review dataset: {json.dumps(business_refs, indent=2)}\n"
        f"- The second column is `business_id` from a business metadata dataset: {json.dumps(business_ids, indent=2)}\n\n"
        "Each business_ref corresponds to a business_id, but the mapping rule is not provided.\n"
        "Determine the mapping relationship between the two sets of IDs.\n"
        "Please respond with:\n"
        "1. The exact mapping rule in plain English\n"
        "2. An explanation of why you think this rule holds."
    )
    response = client.chat.completions.create(
        model=deployment_name,
        messages=[
            {"role": "system", "content": "You are a data engineer analyzing two ID columns from different datasets to infer a mapping rule."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0,
        max_tokens=500
    )
    return response.choices[0].message.content.strip()

mapping_rule_explanation = get_mapping_rule(business_ids, business_refs)
print("\n🧠 GPT-inferred mapping rule:\n")
print(mapping_rule_explanation)

# ========== Step 3: 应用推理规则，将每个 business_ref 映射为 business_id ==========
def resolve_ref_to_id(business_ref):
    prompt = (
        f"The previously inferred mapping rule is:\n\n{mapping_rule_explanation}\n\n"
        f"Now, based on this rule, please determine the corresponding `business_id` for the following `business_ref`:\n"
        f"- business_ref: {business_ref}\n\n"
        "Respond only with the `business_id`, e.g., 'biz_123'."
    )
    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You are a data assistant that maps review business_refs to true business_ids using the inferred rule."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=20
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"❌ GPT error on {business_ref}: {e}")
        return None

# ========== Step 4: 遍历所有 review 记录，逐条转换 ==========
df_review = con_duck.execute("SELECT * FROM review").fetchdf()

predicted_business_ids = []
for i, row in df_review.iterrows():
    business_ref = row["business_ref"]
    business_id = resolve_ref_to_id(business_ref)
    predicted_business_ids.append(business_id)
    print(f"[{i}] business_ref = {business_ref} → business_id = {business_id}")

df_review["business_id"] = predicted_business_ids


In [ ]:
from pymongo import MongoClient

client_mongo = MongoClient("mongodb://localhost:27017/")
biz_collection = client_mongo["yelp_business"]["business"]

# 通过 address 或 description 字段模糊匹配 "Indianapolis"
indianapolis_businesses = biz_collection.find({
    "$or": [
        {"address": {"$regex": "indianapolis", "$options": "i"}},
        {"description": {"$regex": "indianapolis", "$options": "i"}}
    ]
})

# 提取符合条件的 business_id
ind_biz_ids = [biz["business_id"] for biz in indianapolis_businesses if "business_id" in biz]
# df_review 是你之前匹配过 business_id 的完整 review 表
df_ind_reviews = df_review[df_review["business_id"].isin(ind_biz_ids)].copy()

# 检查是否有 rating 字段，计算平均值
if "rating" in df_ind_reviews.columns:
    avg_rating = df_ind_reviews["rating"].mean()
    print(f"📊 Indianapolis 地区商家的平均 rating 为：{avg_rating}")
else:
    print("⚠️ rating 字段不存在，请确认字段名称是否正确")



In [ ]:
from pymongo import MongoClient
from openai import AzureOpenAI
import pandas as pd

client = MongoClient("mongodb://localhost:27017/")
db = client["yelp_business"]
collection = db["business"]

business_docs = list(collection.find())  # 你也可以用分页处理
client = AzureOpenAI(
    api_key="609ced4d971240b8a08f7fb0e6d846ea",
    api_version="2024-08-01-preview",
    azure_endpoint="https://promptdelta-nc.openai.azure.com",
)
deployment_name = "gpt-4o-mini"

def extract_us_state(description):
    prompt = (
        "Based on the following business description, identify the U.S. state where this business is most likely located.\n"
        "Only respond with the full state name (e.g., 'California', 'Texas', 'New York'). If uncertain, respond with 'Unknown'.\n\n"
        f"Description: {description}"
    )

    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You are a helpful assistant that extracts U.S. state names from business descriptions."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=10
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"❌ GPT error: {e}")
        return "Unknown"
# business_docs 已包含 business_id 和 description
state_records = []

for i, biz in enumerate(business_docs):
    desc = biz.get("description", "")
    if not desc:
        continue

    state = extract_us_state(desc)
    biz_id = biz.get("business_id")

    state_records.append({"business_id": biz_id, "state": state})
    print(f"[{i}] {desc} → {state}")
# 构建 state → business_id 的映射 DataFrame
df_state_map = pd.DataFrame(state_records)


In [ ]:
import json
import duckdb
from pymongo import MongoClient
from openai import AzureOpenAI

deployment_name = "gpt-4o-mini"
client = AzureOpenAI(
    api_key="609ced4d971240b8a08f7fb0e6d846ea",
    api_version="2024-08-01-preview",
    azure_endpoint="https://promptdelta-nc.openai.azure.com"
)

# ========== Step 1: 加载两个 DuckDB 表中的 ID ==========
# business_ref from yelp_user.db
con_duck = duckdb.connect("../query_dataset/yelp_user.db")
business_refs = con_duck.execute("SELECT DISTINCT business_ref FROM review").fetchdf()["business_ref"].dropna().tolist()

# ========== Step 2: 从 MongoDB 读取 business_id ==========
client_mongo = MongoClient("mongodb://localhost:27017/")
business_ids = client_mongo["yelp_business"]["business"].distinct("business_id")
# === Step 2: Load business_ref (DuckDB) and business_id (MongoDB) ===
unique_business_refs = sorted(
    con_duck.execute("SELECT DISTINCT business_ref FROM review").fetchdf()["business_ref"].dropna().tolist()
)

client_mongo = MongoClient("mongodb://localhost:27017/")
all_business_ids = sorted(
    client_mongo["yelp_business"]["business"].distinct("business_id")
)

# === Step 3: Infer mapping rule using GPT ===
def get_mapping_rule(business_ids, business_refs):
    prompt = (
        "You are given two complete ID columns from two different datasets:\n"
        f"- The first column is `business_ref` from a review dataset: {json.dumps(business_refs, indent=2)}\n"
        f"- The second column is `business_id` from a business metadata dataset: {json.dumps(business_ids, indent=2)}\n\n"
        "Each business_ref corresponds to a business_id, but the mapping rule is not provided.\n"
        "Determine the mapping relationship between the two sets of IDs.\n"
        "Please respond with:\n"
        "1. The exact mapping rule in plain English\n"
        "2. An explanation of why you think this rule holds."
    )
    response = client.chat.completions.create(
        model=deployment_name,
        messages=[
            {"role": "system", "content": "You are a data engineer analyzing two ID columns from different datasets to infer a mapping rule."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0,
        max_tokens=500
    )
    return response.choices[0].message.content.strip()

mapping_rule_explanation = get_mapping_rule(all_business_ids, unique_business_refs)
print("\n🧠 GPT-inferred mapping rule:\n")
print(mapping_rule_explanation)

# === Step 4: Map business_ref → business_id with caching ===
def resolve_ref_to_id(business_ref):
    prompt = (
        f"The previously inferred mapping rule is:\n\n{mapping_rule_explanation}\n\n"
        f"Now, based on this rule, please determine the corresponding `business_id` for the following `business_ref`:\n"
        f"- business_ref: {business_ref}\n\n"
        "Respond only with the `business_id`, e.g., 'biz_123'."
    )
    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You are a data assistant that maps review business_refs to true business_ids using the inferred rule."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=20
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"❌ GPT error on {business_ref}: {e}")
        return None

# Read full review table
df_review = con_duck.execute("SELECT * FROM review").fetchdf()

# Create cache to avoid duplicate GPT calls
ref_to_id_cache = {}
predicted_business_ids = []

for i, row in df_review.iterrows():
    business_ref = row["business_ref"]
    if pd.isna(business_ref):
        predicted_business_ids.append(None)
        continue

    if business_ref not in ref_to_id_cache:
        business_id = resolve_ref_to_id(business_ref)
        ref_to_id_cache[business_ref] = business_id
    else:
        business_id = ref_to_id_cache[business_ref]

    predicted_business_ids.append(business_id)
    print(f"[{i}] business_ref = {business_ref} → business_id = {business_id}")

df_review["business_id"] = predicted_business_ids


In [ ]:
import pandas as pd

# Merge state info into review data
df_review_with_state = pd.merge(df_review, df_state_map, on="business_id", how="left")

# Drop reviews with unknown or missing state
df_review_with_state = df_review_with_state.dropna(subset=["state"])

# Group by state to count reviews and compute average rating
df_state_stats = df_review_with_state.groupby("state").agg(
    num_reviews=("rating", "count"),
    avg_rating=("rating", "mean")
).reset_index()

# Sort to find the state with the most reviews
df_state_stats = df_state_stats.sort_values(by="num_reviews", ascending=False)
top_state = df_state_stats.iloc[0]

# Print result
print(f"🏆 State with most reviews: {top_state['state']}")
print(f"📝 Number of reviews: {top_state['num_reviews']}")
print(f"⭐ Average rating from that state: {top_state['avg_rating']:.2f}")


In [ ]:
import json
import duckdb
import pandas as pd
from pymongo import MongoClient
from openai import AzureOpenAI

# ========== Step 1: Setup MongoDB and DuckDB ==========
client_mongo = MongoClient("mongodb://localhost:27017/")
biz_collection = client_mongo["yelp_business"]["business"]

con_duck = duckdb.connect("../query_dataset/yelp_user.db")

# ========== Step 2: Setup Azure OpenAI ==========
client = AzureOpenAI(
    api_key="609ced4d971240b8a08f7fb0e6d846ea",
    api_version="2024-08-01-preview",
    azure_endpoint="https://promptdelta-nc.openai.azure.com",
)
deployment_name = "gpt-4o-mini"

# ========== Step 3: Load business_ref and review table ==========
df_review = con_duck.execute("SELECT * FROM review").fetchdf()
unique_business_refs = df_review["business_ref"].dropna().unique().tolist()

# ========== Step 4: Load business_id and attributes from MongoDB ==========
biz_cursor = biz_collection.find({}, {"business_id": 1, "attributes": 1})
business_docs = list(biz_cursor)
all_business_ids = [doc["business_id"] for doc in business_docs if "business_id" in doc]

# ========== Step 5: GPT - Infer business_ref → business_id mapping ==========
def get_mapping_rule(business_ids, business_refs):
    prompt = (
        "You are given two complete ID columns from two different datasets:\n"
        f"- The first column is `business_ref` from a review dataset: {json.dumps(business_refs, indent=2)}\n"
        f"- The second column is `business_id` from a business metadata dataset: {json.dumps(business_ids, indent=2)}\n\n"
        "Each business_ref corresponds to a business_id, but the mapping rule is not provided.\n"
        "Determine the mapping relationship between the two sets of IDs.\n"
        "Please respond with:\n"
        "1. The exact mapping rule in plain English\n"
        "2. An explanation of why you think this rule holds."
    )
    response = client.chat.completions.create(
        model=deployment_name,
        messages=[
            {"role": "system", "content": "You are a data engineer analyzing ID columns from different datasets to infer a mapping rule."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0,
        max_tokens=500
    )
    return response.choices[0].message.content.strip()

mapping_rule_explanation = get_mapping_rule(all_business_ids, unique_business_refs)
print("\n🧠 Inferred Mapping Rule:\n", mapping_rule_explanation)

def resolve_ref_to_id(business_ref):
    prompt = (
        f"The inferred mapping rule is:\n\n{mapping_rule_explanation}\n\n"
        f"Now determine the business_id for:\n"
        f"- business_ref: {business_ref}\n\n"
        "Only respond with the business_id (e.g., 'biz_001')."
    )
    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You are a data assistant mapping business_ref to business_id."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=20
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"❌ GPT error on {business_ref}: {e}")
        return None

# Map all business_refs in review table
ref_to_id_map = {}
predicted_business_ids = []

for i, row in df_review.iterrows():
    business_ref = row["business_ref"]
    if pd.isna(business_ref):
        predicted_business_ids.append(None)
        continue
    if business_ref not in ref_to_id_map:
        ref_to_id_map[business_ref] = resolve_ref_to_id(business_ref)
    predicted_business_ids.append(ref_to_id_map[business_ref])
    print(f"[{i}] {business_ref} → {ref_to_id_map[business_ref]}")

df_review["business_id"] = predicted_business_ids

# ========== Step 6: GPT - Determine which businesses offer parking ==========
def offers_any_parking(attributes):
    prompt = (
    "Given the following business attributes, does the business offer either Business Parking or Bike Parking?\n"
    "Business Parking is considered available if **any** of the following are True: garage, street, validated, lot, valet.\n"
    "Respond only with 'yes' or 'no'.\n\n"
    "Note: BusinessParking may be a dictionary encoded as a string. Parse it before making a decision.\n\n"
    f"Attributes: {json.dumps(attributes, indent=2)}"
    )

    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You check whether businesses offer parking from attributes."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=5
        )
        return response.choices[0].message.content.strip().lower() == "yes"
    except Exception as e:
        print(f"❌ GPT error on attributes: {e}")
        return False

parking_business_ids = []
for i, biz in enumerate(business_docs):
    biz_id = biz.get("business_id")
    attrs = biz.get("attributes", {})
    if not isinstance(attrs, dict):
        continue
    if offers_any_parking(attrs):
        parking_business_ids.append(biz_id)
        print(f"✅ [{i}] {attrs} → offers Parking")
    else:
        print(f"❌ [{i}] {attrs} → no Parking")

# ========== Step 7: Filter reviews from 2018 ==========
df_review["date"] = pd.to_datetime(df_review["date"], unit="ms")
df_2018 = df_review[df_review["date"].dt.year == 2018].copy()


# ========== Step 8: Answer query ==========
df_2018_with_parking = df_2018[df_2018["business_id"].isin(parking_business_ids)]
num_businesses = df_2018_with_parking["business_id"].nunique()

print(f"\n📊 In 2018, number of businesses reviewed that offered Business or Bike Parking: {num_businesses}")



In [ ]:
import duckdb
import pandas as pd

# Step 1: 连接并读取 review 表中的 date 字段
con = duckdb.connect("../query_dataset/yelp_user.db")
df = con.execute("SELECT date FROM review LIMIT 5").fetchdf()

# Step 2: 显示数据和类型
print("🔎 前5条日期值：")
print(df["date"])

print("\n📦 数据类型：")
print(df["date"].dtype)

# Step 3: 尝试使用 .str.startswith("2018")
print("\n✅ 尝试 str.startswith('2018'):")
try:
    print(df["date"].str.startswith("2018"))
except Exception as e:
    print(f"❌ 失败：{e}")

# Step 4: 尝试转为 datetime 再取年份
print("\n✅ 尝试 pd.to_datetime → year == 2018:")
try:
    df["date_parsed"] = pd.to_datetime(df["date"])
    print(df["date_parsed"].dt.year == 2018)
except Exception as e:
    print(f"❌ 失败：{e}")


In [ ]:
from pymongo import MongoClient
from openai import AzureOpenAI
import pandas as pd

client = MongoClient("mongodb://localhost:27017/")
db = client["yelp_business"]
collection = db["business"]

business_docs = list(collection.find())  # 你也可以用分页处理
client = AzureOpenAI(
    api_key="609ced4d971240b8a08f7fb0e6d846ea",
    api_version="2024-08-01-preview",
    azure_endpoint="https://promptdelta-nc.openai.azure.com",
)
deployment_name = "gpt-4o-mini"

def extract_categories(description):
    prompt = (
        "Based on the following business description, identify one or more likely business categories.\n"
        "Respond with a comma-separated list of categories (e.g., 'Nail Salon, Spa', 'Bar, Restaurant'). "
        "Avoid vague responses like 'business' or 'store'.\n\n"
        f"Description: {description}"
    )

    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You are a helpful assistant that extracts specific business categories from descriptions."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=30
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"❌ GPT error on category extraction: {e}")
        return "Unknown"
category_records = []

for i, biz in enumerate(business_docs):
    desc = biz.get("description", "")
    if not desc:
        continue

    biz_id = biz.get("business_id")
    category_str = extract_categories(desc)

    if category_str.lower() == "unknown":
        continue

    categories = [cat.strip() for cat in category_str.split(",") if cat.strip()]
    for cat in categories:
        category_records.append({"business_id": biz_id, "category": cat})

    print(f"[{i}] {desc} → {category_str}")
df_category_map = pd.DataFrame(category_records)


In [ ]:
import duckdb
import pandas as pd

# ========== Step 1: Connect to DuckDB and Load Review Table ==========
con = duckdb.connect("../query_dataset/yelp_user.db")  # 路径按实际修改
df_review = con.execute("SELECT * FROM review").fetchdf()

# ========== Step 2: 检查 date 列的原始格式 ==========
print("📌 Column types:\n", df_review.dtypes)
print("\n📌 Sample 'date' values:")
print(df_review["date"].head(5).tolist())

# ========== Step 3: 推断并转换 date 格式（保留原始值） ==========
df_review["date"]  # 保留原始值


# ========== Step 4: 检查转换后的结果 ==========
print("\n✅ Parsed date preview:")
print(df_review[["date"]].head(5))


In [ ]:
import duckdb

con = duckdb.connect("../query_dataset/yelp_user.db")
tables = con.execute("SHOW TABLES").fetchall()
print("📋 Available tables in DB:")
for t in tables:
    print("-", t[0])


In [ ]:
con.execute("DESCRIBE review").fetchdf()


In [ ]:
print(df_review["date"].sample(20).tolist())


In [ ]:
from openai import AzureOpenAI
def is_in_2018_by_gpt(time_str, client, deployment_name):
    prompt = (
        "Given the following timestamp, determine whether it falls within the year 2018 — "
        "specifically, between January 1, 2018 (inclusive) and January 1, 2019 (exclusive).\n"
        "Only answer with 'yes' or 'no'.\n\n"
        f"Timestamp: {time_str}"
    )

    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You are a helpful assistant that classifies timestamps by date ranges."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=5
        )
        raw_reply = response.choices[0].message.content.strip()
        return raw_reply.lower().startswith("yes")

    except Exception as e:
        print(f"❌ GPT error on '{time_str}': {e}")
        return False
# ========== Configuration ========== #
client = AzureOpenAI(
    api_key="609ced4d971240b8a08f7fb0e6d846ea",
    api_version="2024-08-01-preview",
    azure_endpoint="https://promptdelta-nc.openai.azure.com",  # 不要加 /v1
)
deployment_name = "gpt-4o-mini" 
# ========== Step 7: Use GPT to filter reviews in 2018 ==========
in_2018_mask = []
for i, row in df_review.iterrows():
    time_str = row["date"]
    if pd.isna(time_str):
        in_2018_mask.append(False)
        continue
    result = is_in_2018_by_gpt(time_str, client, deployment_name)
    in_2018_mask.append(result)
    print(f"[{i}] {time_str} → {'✅ in 2018' if result else '❌ not in 2018'}")

df_2018 = df_review[in_2018_mask].copy()


In [ ]:
# ========== Step 1: Setup ==========
import pandas as pd
from pymongo import MongoClient
import json
import ast
from openai import AzureOpenAI

# MongoDB client
client_mongo = MongoClient("mongodb://localhost:27017/")
biz_collection = client_mongo["yelp_business"]["business"]

# OpenAI client (请根据你自己的方式配置)
from openai import OpenAI
deployment_name = "gpt-4o-mini"
deployment_name_large = "gpt-4o"
client = AzureOpenAI(
    api_key="609ced4d971240b8a08f7fb0e6d846ea",
    api_version="2024-08-01-preview",
    azure_endpoint="https://promptdelta-nc.openai.azure.com"
)

# ========== Step 2: Load All Business Descriptions ==========
business_docs = list(biz_collection.find({}, {"business_id": 1, "description": 1}))
biz_id_to_desc = {b["business_id"]: b.get("description", "") for b in business_docs}

# ========== Step 3: 定义 LLM Category 提取函数 ==========
import pandas as pd
import ast
from IPython.display import display

# ========== Step 1: LLM category extraction ==========
def extract_categories(description):
    prompt = (
        "Given the following business description, extract **all category terms that are explicitly mentioned in the text**.\n"
        "Only include categories that appear **verbatim** (word-for-word) in the description.\n"
        "If multiple categories are present, return a **comma-separated list** (e.g., \"Nail Salon, Spa\").\n"
        "Do **not** guess or generalize; do **not** include any explanation.\n\n"
        f"Description: {description}"
    )
    try:
        res = client.chat.completions.create(
            model=deployment_name_large,
            messages=[
                {"role": "system", "content": "You extract business categories."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=30
        )
        return res.choices[0].message.content.strip()
    except:
        return "Unknown"

# ========== Step 2: Run GPT extraction over all businesses ==========
category_records = []
for biz_id, desc in biz_id_to_desc.items():
    if not desc:
        continue
    cats = extract_categories(desc)
    for cat in [c.strip() for c in cats.split(",") if c.strip()]:
        category_records.append({"business_id": biz_id, "category_llm": cat})
    print(biz_id, "→", cats)

df_llm_category = pd.DataFrame(category_records)

# ========== Step 3: Group all LLM categories for each business ==========
df_llm_grouped = df_llm_category.groupby("business_id")["category_llm"].apply(
    lambda x: [c.strip().lower() for c in x if isinstance(c, str) and c.strip()]
).reset_index(name="llm_categories")

# ========== Step 4: Load Ground Truth and parse categories ==========
df_gt = pd.read_json("../ground_truth_dataset/business_gt.json", lines=True)

def parse_category(cat):
    if not isinstance(cat, str):
        return []
    return [c.strip().lower() for c in cat.split(",") if c.strip()]


df_gt["gt_categories"] = df_gt["categories"].apply(parse_category)
df_gt = df_gt[["business_id", "gt_categories"]]

# ========== Step 5: Compare LLM vs GT ==========
df_compare = pd.merge(df_llm_grouped, df_gt, on="business_id", how="left")

def all_llm_in_gt(llm_cats, gt_cats):
    if not isinstance(llm_cats, list) or not isinstance(gt_cats, list):
        return False
    return all(cat in gt_cats for cat in llm_cats)

def get_missing(llm_cats, gt_cats):
    if not isinstance(llm_cats, list) or not isinstance(gt_cats, list):
        return []
    return [cat for cat in llm_cats if cat not in gt_cats]

df_compare["match"] = df_compare.apply(
    lambda row: all_llm_in_gt(row["llm_categories"], row["gt_categories"]),
    axis=1
)

df_compare["missing_categories"] = df_compare.apply(
    lambda row: get_missing(row["llm_categories"], row["gt_categories"]),
    axis=1
)

# ========== Step 6: Print stats and mismatches ==========
total = len(df_compare)
matched = df_compare["match"].sum()
accuracy = matched / total if total > 0 else 0

print(f"\n✅ Matched: {matched} / {total} ({accuracy:.2%})")

# Show mismatches
df_mismatch = df_compare[df_compare["match"] == False]

for _, row in df_mismatch.iterrows():
    print(f"\n❌ Business ID: {row['business_id']}")
    print(f"LLM Categories: {row['llm_categories']}")
    print(f"GT Categories:  {row['gt_categories']}")
    print(f"→ Missing:      {row['missing_categories']}")

display(df_mismatch)


In [ ]:
import pandas as pd
import ast

# 读取原始 ground truth 文件
df_gt_raw = pd.read_json("../ground_truth_dataset/business_gt.json", lines=True)

# 查看前几行 categories 字段
print("\n=== 示例 categories 字段（前 5 行） ===")
print(df_gt_raw["categories"].head())

# 检查字段类型分布
print("\n=== 字段类型统计 ===")
print(df_gt_raw["categories"].apply(type).value_counts())

# 查看包含 '[' 的示例
print("\n=== 包含 '[' 的前几行示例（可能是 list-as-string） ===")
print(df_gt_raw[df_gt_raw["categories"].astype(str).str.contains("\[")]["categories"].head())

# 尝试解析一个字段
sample = df_gt_raw["categories"].dropna().iloc[0]
print("\n=== 解析示例 ===")
print("原始类型:", type(sample))
print("原始值:", sample)

try:
    parsed = ast.literal_eval(sample) if isinstance(sample, str) else sample
    print("解析后类型:", type(parsed))
    print("解析后内容:", parsed)
except Exception as e:
    print("❌ 解析失败:", e)


In [ ]:
# ========== Step 1: 读取数据库 ==========
import duckdb
import pandas as pd
from pymongo import MongoClient

# 连接 DuckDB 和 MongoDB
con_duck = duckdb.connect("../query_dataset/yelp_user.db")
client_mongo = MongoClient("mongodb://localhost:27017/")
biz_collection = client_mongo["yelp_business"]["business"]

# 读取表
df_user = con_duck.execute("SELECT * FROM user").fetchdf()
df_review = con_duck.execute("SELECT * FROM review").fetchdf()

print(f"✅ df_user: {len(df_user)} rows, columns: {df_user.columns.tolist()}")
print(f"✅ df_review: {len(df_review)} rows, columns: {df_review.columns.tolist()}")


In [ ]:
import pandas as pd
import re

def parse_yelping_since(s):
    if pd.isna(s):
        return pd.NaT
    
    s = str(s).strip()

    # 若是 ISO 格式，如 "2010-09-07 23:24:36"
    iso_pattern = r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"
    if re.match(iso_pattern, s):
        return pd.to_datetime(s, errors="coerce")

    # 尝试解析自然语言格式（如 "October 04, 2009 at 05:59 PM"）
    s = s.replace("at", "").replace(",", "").strip()
    return pd.to_datetime(s, errors="coerce", dayfirst=False)

# 应用函数
df_user["yelping_since_parsed"] = df_user["yelping_since"].apply(parse_yelping_since)

# 检查是否还有未能解析的
failed = df_user[df_user["yelping_since_parsed"].isna()]
print(f"❌ 仍然无法解析的注册时间: {len(failed)} 条")
display(failed[["user_id", "yelping_since"]].head(10))


In [ ]:
def parse_review_time(s):
    if pd.isna(s):
        return pd.NaT
    
    s = str(s).strip()
    
    # ISO 格式
    iso_pattern = r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"
    if re.match(iso_pattern, s):
        return pd.to_datetime(s, errors="coerce")
    
    # 清洗自然语言格式
    s = s.replace("at", "").replace(",", "").strip()
    return pd.to_datetime(s, errors="coerce", dayfirst=False)

# 应用到 df_review
df_review["review_date_parsed"] = df_review["date"].apply(parse_review_time)

# 检查结果
failed_reviews = df_review[df_review["review_date_parsed"].isna()]
print(f"❌ 无法解析的评论时间：{len(failed_reviews)} 条")
display(failed_reviews[["user_id", "date"]].head(10))


In [ ]:
# ========== Step 1: 筛选出注册于2016年的用户 ==========
df_user["yelping_since_parsed"] = df_user["yelping_since"].apply(parse_yelping_since)
df_user_2016 = df_user[df_user["yelping_since_parsed"].dt.year == 2016].copy()

print(f"✅ 注册于2016年的用户数量: {len(df_user_2016)}")

# 构建 user_id → 注册时间 的映射
user_since_map = df_user_2016.set_index("user_id")["yelping_since_parsed"].to_dict()
user_ids_2016 = list(user_since_map.keys())


In [ ]:
# ========== Step 2: 找到这些用户的所有 review ==========
df_review["review_date_parsed"] = df_review["date"].apply(parse_review_time)
df_review_2016_users = df_review[df_review["user_id"].isin(user_ids_2016)].copy()

print(f"✅ 这些用户的评论数量: {len(df_review_2016_users)}")


In [ ]:
# ========== Step 3: 保留评论时间晚于注册时间的 ==========
def is_valid_review(row):
    uid = row["user_id"]
    reg_time = user_since_map.get(uid)
    review_time = row["review_date_parsed"]

    if pd.isna(reg_time) or pd.isna(review_time):
        return False
    return review_time >= reg_time

df_review_2016_users["after_registration"] = df_review_2016_users.apply(is_valid_review, axis=1)
df_review_valid = df_review_2016_users[df_review_2016_users["after_registration"]].copy()

print(f"✅ 评论时间晚于注册时间的记录数: {len(df_review_valid)}")
